## Классы и функции для подготовки данных и обучения

In [1]:
import xml.etree.ElementTree as ET
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
import math
from tqdm import tqdm
import pickle
import json

Переиспользуем парсер из предыдущей лабораторной работы и расширим его. Он выделяет из XML следующие признаки:
- **original**: исходное слово;
- **subpart_of_speech**: часть речи слова;
- **form**: форма слова;
- **genesys**: род и одушевлённость слова;
- **semantics1**, **semantics2**: значение семантики слова;
- **nucleus**: находится ли слово под фразовым ударением;
- **is_capital**: начинается ли слово с заглавной буквы;
- **word**: массив графем слова после нормализации;
- **length**: длина слова после нормализации;
- **allophones**: массив аллофонов слова;
- **fonetic_words_before**: количество фонетических слов в предложении перед словом;
- **fonetic_words_after**: количество фонетических слов в предложении после слова;
- **word_num**: номер слова в предложении;
- **has_pause**: есть ли пауза после слова;
- **sintagma_num**: номер синтагмы слова;
- **pause**: длина паузы после слова;
- **sentence_len**: количество слов в предложении, в котором находится данное слово;
- **prev_word**: предыдущее текущему слово в предложении (нормализованное);
- **next_word**: следующее за текущим слово в предложении (нормализованное);
- **PunktBeg**: знак препинания перед словом;
- **PunktEnd**: знак препинания после слова;
- **EmphBeg**: эмфаза перед словом;
- **EmphEnd**: эмфаза после слова.

In [2]:
class Parser():
    def parse_text(self, xml_text):
        self.sintagma_num = 0
        parsed_words = []
        for child in xml_text:
            if child.tag == "sentence":
                parsed_words.extend(self.parse_sentence(child))

        return parsed_words

    def parse_sentence(self, xml_sentence):
        sentence_words = []
        words_count = 0
        fonetic_words_count = 0
        word = {}
        for i in range(len(xml_sentence)):
            child = xml_sentence[i]
            if child.tag == "word":
                word_info = self.parse_word(child)
                word.update(word_info)
                
                word["fonetic_words_before"] = fonetic_words_count
                word["word_num"] = words_count + 1
                word["has_pause"] = False
                word["sintagma_num"] = self.sintagma_num
                words_count += 1
                fonetic_words_count += self.check_if_fonetic(child)
                
                sentence_words.append(word)
                word = {}
            elif child.tag == "content":
                content_info = self.parse_content(child)
                word.update(content_info)
            elif child.tag == "pause":
                self.sintagma_num += 1
                sentence_words[-1]["has_pause"] = True
                if "time" in child.attrib:
                    sentence_words[-1]["pause"] = child.attrib["time"]

        for i in range(len(sentence_words)):
            sentence_words[i]["fonetic_words_after"] = fonetic_words_count -  sentence_words[i]["fonetic_words_before"] - 1
            sentence_words[i]["sentence_len"] = words_count
            sentence_words[i]["prev_word"] = sentence_words[i - 1]["word"] if i > 0 else None
            sentence_words[i]["next_word"] = sentence_words[i + 1]["word"] if i < len(sentence_words) - 1 else None

        return sentence_words
                
    def parse_word(self, xml_word):
        word_info = {}
        if "original" in xml_word.attrib:
            self.original = xml_word.attrib["original"]
            word_info["original"] = self.original
        else:
            word_info["original"] = ""
        
        dictitem = xml_word.find("dictitem")
        if dictitem is not None:
            self.__add_item(word_info, dictitem, "subpart_of_speech")
            self.__add_item(word_info, dictitem, "form")
            self.__add_item(word_info, dictitem, "genesys")
            self.__add_item(word_info, dictitem, "semantics1")
            self.__add_item(word_info, dictitem, "semantics2")

        word_info["nucleus"] = ("nucleus" in xml_word.attrib)
        word_info["is_capital"] = False
        word = []
        allophones = []
        for child in xml_word:
            if child.tag == "letter":
                if ("flag" in child.attrib) and (int(child.attrib["flag"]) & 16 > 0):
                    word_info["is_capital"] = True
                word.append(child.attrib["char"])
            if child.tag == "allophone":
                allophones.append(child.attrib["ph"])

        word_info["word"] = word
        word_info["length"] = len(word_info["word"])
        word_info["allophones"] = allophones

        return word_info

    def check_if_fonetic(self, xml_word):
        for child in xml_word:
            if (child.tag == "letter") and ("flag" in child.attrib) and (int(child.attrib["flag"]) & 1 > 0):
                    return True

        return False

    def parse_content(self, xml_content):
        content_info = {}
        self.__add_item(content_info, xml_content, "PunktBeg")
        self.__add_item(content_info, xml_content, "PunktEnd")
        self.__add_item(content_info, xml_content, "EmphBeg")
        self.__add_item(content_info, xml_content, "EmphEnd")
        return content_info
        
    
    def __add_item(self, info, xml_node, key):
        if key in xml_node.attrib:
            info[key] = xml_node.attrib[key]

Определим функцию для предобработки категориальных данных при помощи LabelEncoder:

In [3]:
def transform(encoder, df_part, column_name):
    df_part[column_name] = df_part[column_name].apply(lambda x: x if x in encoder.classes_ else "<x>")
    df_part[column_name] = encoder.transform(df_part[column_name])

Напишем токенизатор для слов и аллофонов:

In [4]:
class TokensEncoder():
    def __init__(self):
        self.tokens_count = 4
        self.tokens_to_ids = {"<PAD>": 0, "<BOS>": 1, "<EOS>": 2, "<UNK>": 3}
        
    
    def fit(self, df_column):
        for index, row in df_column.items():
            for token in row:
                if token not in self.tokens_to_ids:
                    self.tokens_to_ids[token] = self.tokens_count
                    self.tokens_count += 1

        self.ids_to_tokens = {value: key for key, value in self.tokens_to_ids.items()}

    def transform(self, df_column):
        return df_column.apply(lambda x: [self.tokens_to_ids[token] if token in self.tokens_to_ids else self.tokens_to_ids["<UNK>"] for token in x])

    def inverse_transform(self, df_column):
        return df_column.apply(lambda x: [self.ids_to_tokens[id] for id in x])

Зададим функцию, добавляющую токены начала и конца последовательности:

In [5]:
def add_bos_eos(df_part, column_name):
    df_part[column_name] = df_part[column_name].apply(lambda x: ["<BOS>"] + x + ["<EOS>"])

И функции, удаляющую эти токены:

In [6]:
def remove_bos_eos_from_list(word):
    word.remove("<BOS>")
    word.remove("<EOS>")
    return word

In [7]:
def remove_bos_eos(df_part, column_name):
    df_part[column_name] = df_part[column_name].apply(remove_bos_eos_from_list)

Определим функцию, применяющую TokensEncoder к тестовой и валидационной выборкам:

In [8]:
def train_and_apply_tokens_encoder(tokens_encoder, train_df, val_df, column):
    tokens_encoder.fit(train_df[column])
    train_df[column] = tokens_encoder.transform(train_df[column])
    val_df[column] = tokens_encoder.transform(val_df[column])

Напишем класс датасета, который в первой версии будет использовать только признак **word** и метки **allophones** из DataFrame:

In [9]:
class WordsDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        return torch.LongTensor(row["word"]).to(device), torch.LongTensor(row["allophones"]).to(device)

Определим функтор, реализующий padding:

In [10]:
class WordsPadder():
    def __init__(self, pad_token):
        self.pad_token = pad_token

    def __call__(self, batch):
        words = []
        allophone_sequences = []
        for word, allophone_sequence in batch:
            words.append(word)
            allophone_sequences.append(allophone_sequence)

        padded_words = torch.nn.utils.rnn.pad_sequence(words, padding_value=self.pad_token)
        words_pad_mask = (padded_words == self.pad_token)
        padded_allophone_sequences = torch.nn.utils.rnn.pad_sequence(allophone_sequences, padding_value=self.pad_token)
        allophones_pad_mask = (padded_allophone_sequences == self.pad_token)
        return padded_words, padded_allophone_sequences, words_pad_mask.transpose(-1, -2), allophones_pad_mask.transpose(-1, -2)

Реализуем класс модели, преобразующей слова в аллофоны. Этот класс реализует архитектуру трансформера и использует torch.nn.Embedding в качестве позиционных эмбеддингов.

In [11]:
class Seq2seq(torch.nn.Module):
    def __init__(self, word_tokens_count, allophone_tokens_count, transformer_dim=512, max_seq_len=100, dropout_prob=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.words_emb = torch.nn.Embedding(word_tokens_count, transformer_dim)
        self.words_positional_encoding = torch.nn.Embedding(self.max_seq_len, transformer_dim)
        self.allophones_emb = torch.nn.Embedding(allophone_tokens_count, transformer_dim)
        self.allophones_positional_encoding = torch.nn.Embedding(self.max_seq_len, transformer_dim)
        self.dropout = torch.nn.Dropout(dropout_prob)
        self.transformer = torch.nn.Transformer(d_model=transformer_dim, dropout=dropout_prob)
        self.linear = torch.nn.Linear(transformer_dim, allophone_tokens_count)
    
    def forward(self, words, allophones, words_pad_mask, allophones_pad_mask):
        words = self.words_emb(words)
        words += self.words_positional_encoding(torch.arange(words.shape[0]).to(device)).unsqueeze(1)
        words = self.dropout(words)
        
        allophones = self.allophones_emb(allophones)
        allophones += self.allophones_positional_encoding(torch.arange(allophones.shape[0]).to(device)).unsqueeze(1)
        allophones = self.dropout(allophones)
        result = self.transformer(words, allophones, tgt_mask=torch.nn.Transformer.generate_square_subsequent_mask(allophones.shape[0]).to(device), src_key_padding_mask=words_pad_mask, tgt_key_padding_mask=allophones_pad_mask)
        return self.linear(result)

    def word_to_allophones(self, word):
        allophones = torch.ones((1, 1)).to(device).long()
        while allophones[-1, 0] != 2 and len(allophones) < self.max_seq_len:
            result = self.forward(word, allophones, None, (allophones == 0).transpose(-1, -2))
            allophones = torch.cat([allophones, result.argmax(dim=-1)[result.shape[0] - 1, :].unsqueeze(1)], dim=0)

        return allophones

Определим функцию, реализующую тренировочные и валидационные шаги обучения:

In [12]:
def train_stage(model, optimizer, criterion, dl, stage_name, is_grad_needed=True):
    total_loss = 0
    print(f"{stage_name}:")
    for words, allophones, words_pad_mask, allophones_pad_mask in tqdm(dl):
        if is_grad_needed:
            optimizer.zero_grad()
        results = model(words, allophones[:-1], words_pad_mask, allophones_pad_mask[:, :-1])
        loss = criterion(results.view(-1, allophones_tokens_encoder.tokens_count), allophones[1:].view(-1))
        if is_grad_needed:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
    avg_loss = total_loss / len(dl)
    print("\tAvg loss:", avg_loss)
    return avg_loss

Функцию, реализующую подсчёт метрики Word Recognition Rate:

In [13]:
def pred_stage(model, dl, stage_name):
    print(f"{stage_name}:")
    some_list = []
    for word, allophones, words_pad_mask, allophones_pad_mask in tqdm(dl):
        some_list.append((allophones.view(-1).cpu().numpy(), model.word_to_allophones(word).view(-1).cpu().numpy()))

    some_df = pd.DataFrame(some_list)
    some_df[0] = allophones_tokens_encoder.inverse_transform(some_df[0])
    some_df[1] = allophones_tokens_encoder.inverse_transform(some_df[1])
    print("\tWRR:", (some_df[0] == some_df[1]).value_counts()[True] / len(some_df))

И функцию, реализующую само обучение:

In [14]:
def train(model, optimizer, criterion, train_dl, val_dl, val_dl_for_pred, num_epochs, pred_epoch=5):
    best_val_loss = 10000.0
    for epoch_num in range(1, num_epochs + 1):
        print(f"Epoch {epoch_num}/{num_epochs}")
        model.train()
        train_stage(model, optimizer, criterion, train_dl, "Train")
        with torch.no_grad():
            model.eval()
            val_loss = train_stage(model, optimizer, criterion, val_dl, "Validation", False)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model, "best_model.pt")
            if epoch_num % pred_epoch == 0:
                pred_stage(model, val_dl_for_pred, "Calculating metrics")

## Обучение модели

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

Применим парсер к обучающему xml-файлу:

In [16]:
xml = ET.parse("Kuprin_Aleksandr__Olesya.Result.xml")
xml_root = xml.getroot()
parser = Parser()
df = pd.DataFrame(parser.parse_text(xml_root))
df

,original,subpart_of_speech,form,genesys,semantics2,nucleus,is_capital,word,length,allophones,...,pause,fonetic_words_after,sentence_len,prev_word,next_word,PunktEnd,semantics1,PunktBeg,EmphBeg,EmphEnd
0,А.,1,0,0,700,False,True,[а],1,[a0],...,1,2,3,None,[и],NaN,NaN,NaN,NaN,NaN
1,И.,1,0,0,700,False,True,[и],1,[i0],...,1,1,3,[а],"[к, у, п, р, и, н]",NaN,NaN,NaN,NaN,NaN
2,Куприн,1,1,1,NaN,True,True,"[к, у, п, р, и, н]",6,"[k, u1, p, r', i0, n]",...,359,0,3,[и],None,1,20,NaN,NaN,NaN
3,Олеся,1,1,2,1,True,True,"[о, л, е, с, я]",5,"[a1, l', e0, s', a4]",...,1657,0,1,None,None,1,10,NaN,NaN,NaN
4,I,1,0,0,700,True,True,"[а, й]",2,"[a0, j]",...,644,0,1,None,None,1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22247,любви.,1,2,5,NaN,True,False,"[л, ю, б, в, и]",5,"[l', u1, b, v', i0]",...,800,0,27,"[в, е, л, и, к, о, д, у, ш, н, о, й]",None,1,NaN,NaN,NaN,NaN
22248,< 1898 >,0,1,0,NaN,False,False,"[т, ы, с, я, ч, а]",6,"[t, y0, s', i4, ch, a4]",...,NaN,3,4,None,"[в, о, с, е, м, ь, с, о, т]",NaN,NaN,NaN,5,NaN
22249,,0,1,0,NaN,False,False,"[в, о, с, е, м, ь, с, о, т]",9,"[v, a2, s', i1, m, s, o0, d]",...,NaN,2,4,"[т, ы, с, я, ч, а]","[д, е, в, я, н, о, с, т, о]",NaN,NaN,NaN,NaN,NaN
22250,,0,1,0,NaN,False,False,"[д, е, в, я, н, о, с, т, о]",9,"[d', i1, v', i1, n, o0, s, t, a4]",...,NaN,1,4,"[в, о, с, е, м, ь, с, о, т]","[в, о, с, е, м, ь]",NaN,NaN,NaN,NaN,NaN


Разделим выборку на обучающую и валидационную:

In [17]:
train_df, val_df = train_test_split(df, test_size = 0.2)

Предобработаем массивы слов и аллофонов:

In [18]:
add_bos_eos(train_df, "word")
add_bos_eos(val_df, "word")
add_bos_eos(train_df, "allophones")
add_bos_eos(val_df, "allophones")

In [19]:
word_tokens_encoder = TokensEncoder()
train_and_apply_tokens_encoder(word_tokens_encoder, train_df, val_df, "word")

In [20]:
allophones_tokens_encoder = TokensEncoder()
train_and_apply_tokens_encoder(allophones_tokens_encoder, train_df, val_df, "allophones")

Создадим датасеты и загрузчики данных для обучения, валидации и подсчёта метрики (так как подсчёт метрики занимает наиболее длительное время, для него было выделено подмножества из 1000 примеров из валидационной выборки):

In [21]:
train_ds = WordsDataset(train_df)
val_ds = WordsDataset(val_df)
val_ds_for_pred = WordsDataset(val_df.sample(1000))

In [22]:
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=42, collate_fn=WordsPadder(word_tokens_encoder.tokens_to_ids["<PAD>"]), shuffle=True)
val_dl = torch.utils.data.DataLoader(val_ds, batch_size=42, collate_fn=WordsPadder(word_tokens_encoder.tokens_to_ids["<PAD>"]), shuffle=False)
val_dl_for_pred = torch.utils.data.DataLoader(val_ds_for_pred, batch_size=1, collate_fn=WordsPadder(word_tokens_encoder.tokens_to_ids["<PAD>"]), shuffle=False)

Обучим модель Seq2seq с размерностью пространства эмбеддингов 512. В качестве оптимизатора будем использовать torch.optim.Adam с коэффициентом скорости обучения 0,0001. В качестве критерия будем использовать кросс-энтропию, игнорирующую сравнение для padding токена:

In [23]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [24]:
model = Seq2seq(word_tokens_encoder.tokens_count, allophones_tokens_encoder.tokens_count, 512).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = torch.nn.CrossEntropyLoss(ignore_index=word_tokens_encoder.tokens_to_ids["<PAD>"])
train(model, optimizer, criterion, train_dl, val_dl, val_dl_for_pred, 50, 5)

D:\Пользователи\den26\.pip\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch 1/50
Train:


  0%|                                                                                          | 0/424 [00:00<?, ?it/s]D:\Пользователи\den26\.pip\torch\nn\functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
100%|████████████████████████████████████████████████████████████████████████████████| 424/424 [01:37<00:00,  4.36it/s]


	Avg loss: 1.5557811844320792
Validation:


100%|████████████████████████████████████████████████████████████████████████████████| 106/106 [00:06<00:00, 15.35it/s]


...

	Avg loss: 0.05312893269056419
Validation:


100%|████████████████████████████████████████████████████████████████████████████████| 106/106 [00:07<00:00, 14.68it/s]


	Avg loss: 0.08128711496005361
Calculating metrics:


100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:24<00:00, 11.90it/s]

	WRR: 0.785


In [25]:
best_model = torch.load("best_model.pt", weights_only=False)
best_model.eval()
pred_stage(best_model, val_dl_for_pred, "Calculating metrics for best model")

Calculating metrics for best model:


100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:55<00:00,  8.64it/s]

	WRR: 0.744


Результат модели на 50 эпохе оказался лучше результата модели с наименьшим loss на валидации. Модель показала достаточно неплохое значение метрики. Для улучшения результата можно попробовать обучить большее количество эпох, а также добавить неиспользуемые данной моделью признаки (например, при помощи механизма внимания (cross-attention)).

Сохраним обученную модель и токенизаторы для слов и аллофонов:

In [26]:
torch.save(model, "model.pt")
with open("word_tokens_encoder.pickle", "wb") as file:
    pickle.dump(word_tokens_encoder, file)

with open("allophones_tokens_encoder.pickle", "wb") as file:
    pickle.dump(allophones_tokens_encoder, file)

## Предсказание аллофонов и формирование JSON

Загрузим обученную модель и токенизаторы:

In [39]:
loaded_model = torch.load("model.pt", weights_only=False)
model.eval()

with open("word_tokens_encoder.pickle", "rb") as file:
    loaded_word_tokens_encoder = pickle.load(file)

with open("allophones_tokens_encoder.pickle", "rb") as file:
    loaded_allophones_tokens_encoder = pickle.load(file)

Применим парсер к тестовому XML-файлу:

In [40]:
test_xml = ET.parse("Example_Input_Phonetics.xml")
test_xml_root = test_xml.getroot()
test_parser = Parser()
test_df = pd.DataFrame(parser.parse_text(test_xml_root))
test_df

,original,subpart_of_speech,form,genesys,nucleus,is_capital,word,length,allophones,fonetic_words_before,word_num,has_pause,sintagma_num,fonetic_words_after,sentence_len,prev_word,next_word,semantics2,PunktEnd,pause
0,Больше,10,0,0,False,True,"[б, о, л, ь, ш, е]",6,[],0,1,False,0,10,19,None,"[в, с, е, г, о]",NaN,NaN,NaN
1,всего,1,2,6,False,False,"[в, с, е, г, о]",5,[],1,2,False,0,9,19,"[б, о, л, ь, ш, е]","[ч, е, л, о, в, е, к]",2,NaN,NaN
2,человек,1,1,1,False,False,"[ч, е, л, о, в, е, к]",7,[],2,3,False,0,8,19,"[в, с, е, г, о]","[н, е, н, а, в, и, д, и, т]",NaN,NaN,NaN
3,ненавидит,6,63,0,False,False,"[н, е, н, а, в, и, д, и, т]",9,[],3,4,False,0,7,19,"[ч, е, л, о, в, е, к]","[т, е, х]",10,NaN,NaN
4,"тех,",3,40,0,True,False,"[т, е, х]",3,[],4,5,True,0,6,19,"[н, е, н, а, в, и, д, и, т]","[к, т, о]",3,2,NaN
5,кто,1,1,1,False,False,"[к, т, о]",3,[],5,6,False,1,5,19,"[т, е, х]","[с, м, е, ё, т, с, я]",2,NaN,NaN
6,смеется,6,63,0,False,False,"[с, м, е, ё, т, с, я]",7,[],5,7,False,1,5,19,"[к, т, о]","[е, м, у]",NaN,NaN,NaN
7,ему,11,4,1,False,False,"[е, м, у]",3,[],6,8,False,1,4,19,"[с, м, е, ё, т, с, я]","[п, р, я, м, о]",2,NaN,NaN
8,прямо,10,0,0,False,False,"[п, р, я, м, о]",5,[],6,9,False,1,4,19,"[е, м, у]",[в],NaN,NaN,NaN
9,в,9,0,0,False,False,[в],1,[],7,10,False,1,3,19,"[п, р, я, м, о]","[г, л, а, з, а]",100,NaN,NaN


Предобработаем массивы слов:

In [41]:
add_bos_eos(test_df, "word")
test_df["word"] = loaded_word_tokens_encoder.transform(test_df["word"])

In [42]:
test_ds = WordsDataset(test_df)
test_dl = torch.utils.data.DataLoader(test_ds, batch_size=1, collate_fn=WordsPadder(word_tokens_encoder.tokens_to_ids["<PAD>"]), shuffle=False)

Запустим модель для генерации массивов аллофонов:

In [43]:
with torch.no_grad():
    pred_list = []
    for word, allophones, words_pad_mask, allophones_pad_mask in tqdm(test_dl):
        pred_list.append((word.view(-1).cpu().numpy(), loaded_model.word_to_allophones(word).view(-1).cpu().numpy()))

pred_df = pd.DataFrame(pred_list, columns = ["content", "allophones"])
pred_df["content"] = loaded_word_tokens_encoder.inverse_transform(pred_df["content"])
pred_df["allophones"] = loaded_allophones_tokens_encoder.inverse_transform(pred_df["allophones"])
remove_bos_eos(pred_df, "content")
remove_bos_eos(pred_df, "allophones")
pred_df

  0%|                                                                                           | 0/19 [00:00<?, ?it/s]D:\Пользователи\den26\.pip\torch\nn\functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
100%|██████████████████████████████████████████████████████████████████████████████████| 19/19 [00:01<00:00, 14.41it/s]


,content,allophones
0,"[б, о, л, ь, ш, е]","[b, o0, l', sh, y4]"
1,"[в, с, е, г, о]","[f, s', i1, v, o0]"
2,"[ч, е, л, о, в, е, к]","[ch, i1, l, a1, v', e0, k]"
3,"[н, е, н, а, в, и, д, и, т]","[n', i1, n, a1, v', i0, d', i4, t]"
4,"[т, е, х]","[t', e0, h]"
5,"[к, т, о]","[k, t, o0]"
6,"[с, м, е, ё, т, с, я]","[s, m', i1, j, o0, c, a4]"
7,"[е, м, у]","[j, i1, m, u0]"
8,"[п, р, я, м, о]","[p, r', a0, m, a4]"
9,[в],[f]


Сохраним результаты в заданном формате в JSON-файл:

In [45]:
pred_df["content"] = test_df["original"]
words = json.loads(pred_df.to_json(orient="records", force_ascii=False, indent=4))
result_json = [{"words": words}]
with open("result.json", 'w') as result_file:
    json.dump(result_json, result_file, ensure_ascii=False, indent=4)